In [ ]:
# Common imports + DES (Modelica) translator pieces.

from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner
from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

### Activity 4: DES/TEN

In [ ]:
# Copy activity_2/coincident to activity_4/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_2") / "coincident"
activity_4_dir = paths.activity_dir("activity_4")
dest_dir = activity_4_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_4/coincident
uo_coincident = UO(activity_4_dir, "coincident", template_dir=template_data_dir)

# Weather data is copied over from the activity_2/coincident project. No
# need to update the weather information.

In [ ]:
# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Generate sys params with absolute paths to avoid cwd-dependent des_create failures.
sp = SystemParameters()
sp.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=sys_param_path,
    scenario_dir=uo_coincident.project_path / "run" / "baseline_scenario",
    feature_file=feature_path,
    district_type="4G",
    relative_path=None,
    modelica_load_filename="modelica.mos",
    overwrite=True,
)

print(f"Created sys params: {sys_param_path}")

In [ ]:
# Fix values in the sys_params.json file for the 4G system if needed.
# Example overrides:
#   "central_cooling_plant_parameters": {
#     "heat_flow_nominal": 8_000_000,   # 8 MW, was 7999 W
#     "cooling_tower_fan_power_nominal": 50_000,
#     "mass_chw_flow_nominal": 85,      # ~Σ building chw flows
#     "chiller_water_flow_minimum": 20, # ~0.25 × nominal
#     "mass_cw_flow_nominal": 100,      # ~1.15 × chw nominal - prevents freeze
#   }
#
# Build the 4G Modelica model from sys_params + feature file.
des_scenario_name = "four_g"
uo_coincident.des_create(
    sys_param_path, feature_path, des_scenario_name, overwrite=True
)

# UOCliWrapper runs `uo` from activity_4, so relative des_name is created there.
des_model_path = activity_4_dir / des_scenario_name
print(f"Created 4G model at: {des_model_path}")

In [ ]:
# 4G run via the gmt-om-runner. Gated so it does not run accidentally.
RUN_4G = False
des_scenario_name = "four_g"
activity_4_dir = paths.activity_dir("activity_4")
des_analysis_4g_dir = activity_4_dir / des_scenario_name
if RUN_4G:
    print(f"{des_scenario_name}.Districts.district")
    print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
    print(f"run_path={des_analysis_4g_dir}")

    results_path = (
        des_analysis_4g_dir
        / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
    )
    if results_path.exists():
        print("Results already exist, skipping model run.")
    else:
        mr = ModelicaRunner()
        _, results_path = mr.run_in_docker(
            "compile_and_run",
            f"{des_scenario_name}.Districts.DistrictEnergySystem",
            file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
            run_path=des_analysis_4g_dir,
            # March 1 → March 14
            start_time=5097600,
            stop_time=6307200,
        )

        # step_size=300,  # 5-min step
        # stop_time=31_536_000,  # full year

print(f"4G results path: {des_analysis_4g_dir}")